### Combine All Datasets from Human, SARS-CoV-2, and Influenza

für READ ME:
In this notebook, three data sets from three different sources (human, influenza, SARS-CoV-2) are merged and supplemented by an sorce column. The relative frequency of the 20 amino acids is then calculated for each CDR region. The results are saved in all_data.tsv and in all_data_normalozed.tsv and displayed for control purposes.

In [5]:
import pandas as pd

# load files 
df_human = pd.read_csv("../generated/cdrs/seq/human_cdr_seq.tsv", sep="\t")
df_influenza = pd.read_csv("../generated/cdrs/seq/influenza_cdr_seq.tsv", sep="\t")
df_sars = pd.read_csv("../generated/cdrs/seq/corona_cdr_seq.tsv", sep="\t")

# add source
df_human["source"] = "human"
df_influenza["source"] = "influenza"
df_sars["source"] = "sars_cov2"

# combine all data
all_data = pd.concat([df_human, df_influenza, df_sars], ignore_index=True)

# control
print("conclusion:", all_data.shape[0], "rows,", all_data.shape[1], "columns")
print("column name:", list(all_data.columns)[:10], "...")

# save in a new df
all_data.to_csv("../generated/cdrs/seq/all_data.tsv", sep="\t", index=False)
print("saved as: all_data.tsv")


conclusion: 961 rows, 81 columns
column name: ['pdb', 'heavy_chain', 'CDR_H3', 'CDR_H2', 'CDR_H1', 'CDR_H1_count_A', 'CDR_H1_count_R', 'CDR_H1_count_N', 'CDR_H1_count_D', 'CDR_H1_count_C'] ...
saved as: all_data.tsv


### Control View – Preview the First Rows of the Table

In [6]:
# Load the file (TSV = Tab-separated values)
df = pd.read_csv("../generated/cdrs/seq/all_data.tsv", sep="\t")

# Control view: display the first few rows of the table
df.head()



,pdb,heavy_chain,CDR_H3,CDR_H2,CDR_H1,CDR_H1_count_A,CDR_H1_count_R,CDR_H1_count_N,CDR_H1_count_D,CDR_H1_count_C,...,CDR_H2_mean_polarity,CDR_H2_mean_mass,CDR_H3_mean_hydrophobicity,CDR_H3_mean_charge,CDR_H3_mean_polarity,CDR_H3_mean_mass,cdr_h1_length,cdr_h2_length,cdr_h3_length,source
0,8rmx,E,RFVGTLDV,FWDDD,GFSLDTSGV,0.0,0.0,0.0,1.0,0.0,...,9.920000,135.734000,0.73750,0.0000,7.87500,113.255000,9,5,8,human
1,8rmy,E,RFVGTLDV,FWDDD,GFSLDTSGV,0.0,0.0,0.0,1.0,0.0,...,9.920000,135.734000,0.73750,0.0000,7.87500,113.255000,9,5,8,human
2,8znz,F,AIYYYGSSYNYYAMDY,SSGGGY,GFTFSKY,0.0,0.0,0.0,0.0,0.0,...,8.600000,93.748333,-0.50625,-0.0625,7.65625,125.449375,7,6,16,human
3,9avo,B,GYGWALDY,YPYYGS,GFNVSYY,0.0,0.0,1.0,0.0,0.0,...,7.466667,124.798333,-0.27500,-0.1250,7.72500,120.251250,7,6,8,human
4,9awe,B,GYGWALDY,YPYYGS,GFNVSYY,0.0,0.0,1.0,0.0,0.0,...,7.466667,124.798333,-0.27500,-0.1250,7.72500,120.251250,7,6,8,human


#### Erklärung Code:

1. Bibliothek laden**

- Bibliothek `pandas` laden (mit der man Tabellen (DataFrames) verarbeiten kann)
- `pd` ist die übliche Kurzform (Alias) für `pandas`.


2. Drei Dateien laden**

- `pd.read_csv(...)`: Liest eine Datei ein, auch wenn sie `.tsv` heißt 
- `sep="\t"`: Sagt `pandas`, dass die Daten **mit Tabs getrennt** sind (nicht mit Kommas)
- `"../data cleanup/..."`: Ist ein Pfad zur Datei **aus einem übergeordneten Ordner**.


3. Quelle hinzufügen**

-  eine neue Spalte ("source") hinzufügen


4. Alle Daten kombinieren**

- `pd.concat([...])`: Hängt die drei Tabellen **untereinander**.
- `ignore_index=True`: Die Zeilennummern (Index) werden **neu durchgezählt**.
=  eine neue Tabelle `all_data`


5. Kontrolle: Form & Spaltennamen**

- `all_data.shape[0]`: Anzahl der Zeilen
- `all_data.shape[1]`: Anzahl der Spalten
- `list(all_data.columns)[:10]`: Zeigt dir die **ersten 10 Spaltennamen**.



6. Neue Datei speichern**

- `to_csv(...)`: Speichert die Daten in eine neue Datei.
- `"all_data.tsv"`: Der Name der neuen Datei.
- `sep="\t"`: Wieder als **TSV-Datei** (mit Tabs).
- `index=False`: Die Pandas-Zeilennummern werden **nicht gespeichert**.




#### für Read ME
summary_all_data.ipynb

Zusammenführen von CDR-Sequenzdaten

Dieses Skript macht Folgendes:

- Lädt drei Datensätze mit Antikörper-Informationen (jeweils gegen:
humane Strukturen (human), Influenza-Viren (influenza), SARS-CoV-2 (sars_cov2)).

- Fügt eine neue Spalte hinzu, die angibt, woher die Daten kommen (z. B. "source" = human).

- Verbindet alle drei Tabellen zu einer großen gemeinsamen Tabelle.

- Zeigt an, wie viele Zeilen und Spalten die neue Tabelle hat, und gibt einige Spaltennamen aus.

- Speichert die neue Tabelle unter dem Namen all_data.tsv als Datei (im aktuellen Ordner).


###  Normalize Amino Acid Counts by CDR Length to Get Relative Frequencies

In [7]:
#Calculate relative amino acid abundance

# Load the merged dataset
df = pd.read_csv("../generated/cdrs/seq/all_data.tsv", sep="\t")

# List of the 20 amino acids
aminos = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I',
          'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

# Calculate relative frequency for each CDR region
for cdr in ['CDR_H1', 'CDR_H2', 'CDR_H3']: # outer loop 3x
    length_col = f"{cdr.lower()}_length" 

    for aa in aminos: # aa new variable, inner loop
        count_col = f"{cdr}_count_{aa}"      # column names of df 
        rel_col = f"{cdr}_relfreq_{aa}"      # creation of new column for relative frequency

        if count_col in df.columns and length_col in df.columns:
            df[rel_col] = df[count_col] / df[length_col] # The new column is created here: Each cell is: number of this amino acid / length of CDR

# Save result to a new file (optional)
df.to_csv("../generated/cdrs/seq/all_data_normalized.tsv", sep="\t", index=False)
print("Saved normalized file as: all_data_normalized")

Saved normalized file as: all_data_normalized


### Control View – Preview the First Rows of the new Table

In [8]:
all_data_normalized = pd.read_csv("../generated/cdrs/seq/all_data_normalized.tsv", sep="\t")
all_data_normalized.head(10)


,pdb,heavy_chain,CDR_H3,CDR_H2,CDR_H1,CDR_H1_count_A,CDR_H1_count_R,CDR_H1_count_N,CDR_H1_count_D,CDR_H1_count_C,...,CDR_H3_relfreq_L,CDR_H3_relfreq_K,CDR_H3_relfreq_M,CDR_H3_relfreq_F,CDR_H3_relfreq_P,CDR_H3_relfreq_S,CDR_H3_relfreq_T,CDR_H3_relfreq_W,CDR_H3_relfreq_Y,CDR_H3_relfreq_V
0,8rmx,E,RFVGTLDV,FWDDD,GFSLDTSGV,0.0,0.0,0.0,1.0,0.0,...,0.125000,0.0,0.0000,0.125000,0.000000,0.000000,0.125000,0.000000,0.000000,0.25
1,8rmy,E,RFVGTLDV,FWDDD,GFSLDTSGV,0.0,0.0,0.0,1.0,0.0,...,0.125000,0.0,0.0000,0.125000,0.000000,0.000000,0.125000,0.000000,0.000000,0.25
2,8znz,F,AIYYYGSSYNYYAMDY,SSGGGY,GFTFSKY,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0625,0.000000,0.000000,0.125000,0.000000,0.000000,0.437500,0.00
3,9avo,B,GYGWALDY,YPYYGS,GFNVSYY,0.0,0.0,1.0,0.0,0.0,...,0.125000,0.0,0.0000,0.000000,0.000000,0.000000,0.000000,0.125000,0.250000,0.00
4,9awe,B,GYGWALDY,YPYYGS,GFNVSYY,0.0,0.0,1.0,0.0,0.0,...,0.125000,0.0,0.0000,0.000000,0.000000,0.000000,0.000000,0.125000,0.250000,0.00
5,9dq3,H,TGWLGPFDY,SYDGRH,GFTFSKY,0.0,0.0,0.0,0.0,0.0,...,0.111111,0.0,0.0000,0.111111,0.111111,0.000000,0.111111,0.111111,0.111111,0.00
6,9gox,H,SRGYYGSFYFDI,IPVIDD,GGFFSHY,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0000,0.166667,0.000000,0.166667,0.000000,0.000000,0.250000,0.00
7,9goy,H,GLGYYLYSSYYFDI,IPRLDA,GLPDATY,1.0,0.0,0.0,1.0,0.0,...,0.142857,0.0,0.0000,0.071429,0.000000,0.142857,0.000000,0.000000,0.357143,0.00
8,9l1s,C,NLGPSFYFDY,NPNSGG,GFTFSAY,1.0,0.0,0.0,0.0,0.0,...,0.100000,0.0,0.0000,0.200000,0.100000,0.100000,0.000000,0.000000,0.200000,0.00
9,8t6m,B,TTGYCSGGSCYSGWFDP,YWDHN,GFSVNTSGV,0.0,0.0,1.0,0.0,0.0,...,0.000000,0.0,0.0000,0.058824,0.058824,0.176471,0.117647,0.058824,0.117647,0.00


### Erklärung

#### Relative Aminosäurehäufigkeit berechnen

Dieses Skript berechnet **relative Häufigkeiten** von Aminosäuren in den drei CDR-Regionen (CDR\_H1, CDR\_H2, CDR\_H3) für jeden Antikörper.

Statt nur die absolute Anzahl zu betrachten (z. B. wie oft „A“ vorkommt), wird berechnet:

> Wie oft kommt Aminosäure im verhältnis zur Länge vor?

Das macht die Daten vergleichbar – unabhängig davon, wie lang die Schleifen (CDRs) sind.


* **Datensatz laden:**
  Die Datei `all_data.tsv` wird als Tabelle (DataFrame) eingelesen.

* **Aminosäurenliste:**
  Es wird mit den 20 natürlichen Aminosäuren gearbeitet (A, R, N, D, …).

* **Für jede CDR-Region (H1, H2, H3):**

  * Für jede Aminosäure:

    * Wird die absolute Häufigkeit (z. B. `CDR_H1_count_A`) durch die Länge der CDR (`cdr_h1_length`) geteilt.
    * Das ergibt eine neue Spalte: `CDR_H1_relfreq_A` (relative Häufigkeit von A in CDR\_H1).

* **Ergebnis speichern:**
  Die neue Tabelle wird als Datei `all_data_normalized.tsv` gespeichert.

---

Eine neue Datei `all_data_normalized.tsv` mit **vielen neuen Spalten**, z. B.:

* `CDR_H1_relfreq_A`
* `CDR_H2_relfreq_G`
* `CDR_H3_relfreq_Y`

Diese Spalten zeigen:
**Wie häufig kommt jede Aminosäure durchschnittlich pro Position in jeder Region vor?**


